# 3.4 Autonomous Flight - Finding the Rubber Duck

## Object Detection Capabilities

In the previous lesson, the drone could identify objects in the environment, but still needed manual operation to approach targets.

Now we need to know the target's position. Previously this was done via the get_position API. In this lesson, we use object detection so the drone can determine the target's position within the camera frame.

New drone function:

detect(object_name)

Input: the target object_name to detect. The function internally calls the detection model on the camera image.

Returns 3 variables:
- obj_list: list of detected object names in the scene.
- obj_locs: list of bounding box coordinates for each object in the image.
- img_with_box: annotated image with bounding boxes.

Note: the object name input must be in English.

Additionally, the drone has the following movement actions:
- forward(): Move forward by 1 meter.
- turn_left(): Turn left by 10 degrees.
- turn_right(): Turn right by 10 degrees.

In [ ]:
from dds_cloudapi_sdk.tasks.v2_task import create_task_with_local_image_auto_resize
from dds_cloudapi_sdk import Config
from dds_cloudapi_sdk import Client
from dds_cloudapi_sdk.visualization_util import visualize_result

def detect(self, object_names):
"""
for imageobject detection, return class, and image
:param object_names: target (string, 'duck, cola') 
:return: obj_id_list, obj_locs, vis_img
"""
config = Config(gdino_token) # use token
client = Client(config)
# Step 1: Read camera image (already RGB)
rgb_image = self.get_image()

# Direct cv image display has bugs on Windows
# Generate random filename (with extension)
file_name = f"random_{uuid.uuid4().hex}.png" # Example output: random_1a2b3c4d5e.png
cv2.imwrite(file_name, rgb_image)
# Create detection task
task = create_task_with_local_image_auto_resize( 
api_path="/v2/task/dinox/detection",
api_body_without_image={
"model": "DINO-X-1.0",
"prompt": {
"type": "text",
"text": object_names
},
"targets": ["bbox"],
"bbox_threshold": 0.25,
"iou_threshold": 0.8
},
image_path=file_name
)
client.run_task(task)
result = task.result

# Parse detection results
obj_id_list = [obj['category'] for obj in result['objects']]
obj_locs = [obj['bbox'] for obj in result['objects']]

# Visualization
try:
visualize_result(image_path=file_name, result=result, output_dir="./")
vis_img = Image.open(file_name) # or use output_dir below image
except Exception as e:
vis_img = None
#os.remove(file_name)

return obj_id_list, obj_locs, vis_img


## Building the Prompt

In [ ]:
# Write knowledge base prompt file aisim_lession34.txt
kg_promot_file = "./prompts/aisim_lession34.txt"

kg_prompt = """
Imagine you are helping me interact with the AirSim simulator.

We are controlling a physical Agent. At any given point in time, you have the following capabilities. You also need to output code for certain requests.
Question - Ask me a clarification question.
Reason - Explain why you did it that way.
Code - Output code commands that achieve the desired goal.

The scene consists of multiple objects. We can use the following functions; please use only these functions whenever possible:

Perception:

aw.detect(object_name): Renders an image from the drone's front camera, runs an object detection model on the image to find object_name, and returns 3 variables:
- obj_list: a list of detected object names in the scene.
- obj_locs: a list of bounding box coordinates for each object in the image.
- img_with_box: an annotated image with bounding boxes, in PngImageFile format. You can get its dimensions (width, height) via img_with_box.size. You can also display it using display(img_with_box), but you need to import first: from IPython.display import display

Note: The object_name input for this function must be in English. If the target name is in Chinese, translate it first.

Actions:
aw.takeoff() - Takes off the drone.
aw.land() - Lands the drone.
aw.forward(): Move forward by 1 meter.
aw.turn_left(): Turn left by 10 degrees.
aw.turn_right(): Turn right by 10 degrees.

When generating code, you do not need to worry about importing aw; it is already declared in the environment.

You must not use any other hypothetical functions. You can use functions from Python libraries such as math, numpy, etc. Are you ready?



"""

pt_file = open(kg_promot_file, "w", encoding="utf-8")
pt_file.write(kg_prompt)
pt_file.close()

## Execution

In [ ]:
import sys
sys.path.append('..')
import airsim
import time

client = airsim.MultirotorClient()
client.confirmConnection()
client.enableApiControl(True)
client.armDisarm(True)

state = client.getMultirotorState()
pos = state.kinematics_estimated.position
print("Takeoff:", pos.x_val, pos.y_val, pos.z_val)

In [ ]:
client = airsim.MultirotorClient()
client.confirmConnection()
client.enableApiControl(True)
client.armDisarm(True)

# Takeoff
client.takeoffAsync().join()
time.sleep(1)

# Safe altitude
client.moveToZAsync(-3, 1).join()
time.sleep(1)

# Fly to the waypoint (point B)
target_x = 9.0
target_y = 16.5
target_z = -1.0

client.moveToPositionAsync(target_x, target_y, target_z, 2).join()
time.sleep(2)

# Hover
client.hoverAsync().join()

print("Arrived above point B")

In [ ]:
# Create an AirSim wrapper instance for execution
import importlib
import airsim_wrapper
importlib.reload(airsim_wrapper)
aw = airsim_wrapper.AirSimWrapper()

In [ ]:
# Create the LLM agent
import airsim_agent
my_agent = airsim_agent.AirSimAgent(knowledge_prompt="prompts/aisim_lession34.txt")

In [ ]:
command = "Take off"
ret = my_agent.process(command, False) # Don't execute code
print("ret:", ret)

In [ ]:
# Manual execution -------

aw.takeoff()

In [ ]:
command = """
I need to find a target. The target might be anywhere in the scene. If it's not visible, a good strategy is to rotate and scan. This is an exploration task to find the rubber duck.
"""
ret = my_agent.process(command, False) # Don't execute code
print("ret:", ret)

In [ ]:
# in target" small "aw.detect input need 
target_name = "yellow duck"
found_target = False

# search360, each times10, each times before toward 
for _ in range(36):
# before toward target
obj_list, _, _ = aw.detect(target_name)
# targetstopsearch
if len(obj_list) > 0:
found_target = True
print("success small target, stopsearch")
break
# 10 toward continuesearch
aw.turn_right()

# searchcompletetargetoutput
if not found_target:
print("search after, small target")

In [ ]:
command = """
Great! We found the rubber duck. Now I want to approach it. The information we have is the duck's bounding box position in the image. Can you implement an approach strategy?
"""
ret = my_agent.process(command, False) # Don't execute code
print("ret:", ret)

In [ ]:
import numpy as np

# false before small, this inside use before variable
obj_list, obj_locs, img_with_box = aw.detect('rubber duck')
print(obj_list)

# small edge index
duck_index = obj_list.index('rubber duck')
duck_bbox = obj_locs[duck_index]

# calculateimage in 
img_width,img_height = img_with_box.size
img_center_x = img_width / 2

# calculate small edge in 
duck_center_x = (duck_bbox[0] + duck_bbox[2]) / 2

# error, small in and image in in this within, drone for small 
error_threshold = 100

for i in range(10):# many 10

# toward 
while abs(duck_center_x - img_center_x) > error_threshold:
print("img_center_x",img_center_x,"duck_center_x",duck_center_x)
if duck_center_x < img_center_x:
# small in image, 
aw.turn_left()
else:
# small in image, 
aw.turn_right()

# times small with updateedgeposition
obj_list, obj_locs, img_with_box = aw.detect('rubber duck')
print(obj_list)

if "rubber duck" in obj_list:
display(img_with_box)
duck_index = obj_list.index('rubber duck')
duck_bbox = obj_locs[duck_index]
duck_center_x = (duck_bbox[0] + duck_bbox[2]) / 2

# drone for small, toward before 
print(" toward, toward before ")
aw.forward()
# times small with updateedgeposition
obj_list, obj_locs, img_with_box = aw.detect('rubber duck')
print(obj_list)
if "rubber duck" in obj_list:
display(img_with_box)
duck_index = obj_list.index('rubber duck')
duck_bbox = obj_locs[duck_index]
duck_center_x = (duck_bbox[0] + duck_bbox[2]) / 2

In [ ]:
aw.set_yaw(0)

As you can see, based on camera-based object detection, the LLM can generate a PID-like control strategy to approach the target (the rubber duck).

The basic principle is as follows:

<img src='img/duck.jpg' width='640px' />

The drone's camera image has its center as the reference point (0,0).

If the duck is on the left side of the image, the drone turns left. If it's on the right side, it turns right.

After aligning, move forward a certain distance, then repeat.

The LLM generates reasonable code, but it needs to be executed iteratively multiple times.

## Homework
In this lesson, we gave the drone object detection and approach capabilities.

What if we also give the drone a take() function and a drop() function?

Then we could tell the drone to "deliver a package."

How would the drone respond to this task?